# NB-4: Ablation Study — 5 Forgetting Strategies (Multi-Model + Multi-Seed)

Compares 5 forgetting strategies with False Forgetting Rate (FFR) — the key metric
proving that consolidation-aware forgetting outperforms all baselines.

**Strategies:** No-Forgetting · LRU · Importance-Only · CA-Formula-Only · CA-Ours (full)  
**Runs:**
- 8B model: seeds 42, 123, 456 (for variance / CI reporting)
- Scout-17B: seed 42
- Llama-3.3-70B: seed 42

**Key metric:** FFR = fraction of evicted memories with consolidation coverage < 0.3  
**Expected:** CA-Ours FFR ≈ 0 · all other strategies FFR > 0

**Output directory:** `results/nb4_ablation/`  
**Time estimate:** ~20–40 min per run (5 strategies × conversations)

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set API keys

**Kaggle:** Secrets → `GROQ_API_KEY` (+ optionally `GROQ_API_KEY_2` … `GROQ_API_KEY_5`)  
**Colab:** Left sidebar key icon → same secrets

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')

## Step 3 — Configure

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
CONVERSATIONS = 5    # conversations per ablation run
INTERACTIONS  = 50   # memory events per conversation (50=fast, 100=paper quality)
THRESHOLD     = 80   # forget trigger: evict when L2 > threshold entries
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR = os.path.join(REPO_DIR, 'results', 'nb4_ablation')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Output dir:   {OUT_DIR}')
print(f'Conversations:{CONVERSATIONS}')
print(f'Interactions: {INTERACTIONS}')
print(f'Threshold:    {THRESHOLD}')
print(f'Approx API calls per run: ~{CONVERSATIONS * 10 * 5} (5 strategies)')

RUNS = [
    ('llama-3.1-8b-instant',              42,  '8b_s42'),
    ('llama-3.1-8b-instant',             123,  '8b_s123'),
    ('llama-3.1-8b-instant',             456,  '8b_s456'),
    ('meta-llama/llama-4-scout-17b-16e-instruct', 42, 'scout17b_s42'),
    ('llama-3.3-70b-versatile',           42,  '70b_s42'),
]

print(f'\nRuns scheduled: {len(RUNS)}')
for model, seed, tag in RUNS:
    out = os.path.join(OUT_DIR, f'ablation_{tag}.json')
    print(f'  [{"EXISTS" if os.path.exists(out) else "PENDING"}] {tag}: {model} seed={seed}')

## Step 4 — Run ablation for all model/seed combinations
Each run tests all 5 strategies on the same conversations for direct comparison.
Progress prints `Sem=X.XXX` for each QA pair — if missing, restart kernel and re-pull.

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

failed_runs = []

for model, seed, tag in RUNS:
    out_path = os.path.join(OUT_DIR, f'ablation_{tag}.json')
    if os.path.exists(out_path):
        print(f'[SKIP] {tag} already exists — delete to re-run')
        continue

    cmd = [
        sys.executable, '-m', 'csam_project.evaluation.run_ablation',
        '--conversations', str(CONVERSATIONS),
        '--interactions',  str(INTERACTIONS),
        '--threshold',     str(THRESHOLD),
        '--model',         model,
        '--seed',          str(seed),
        '--output',        out_path,
    ]
    print(f'\n{"="*60}')
    print(f'Running ablation: {tag} (model={model}, seed={seed})')
    print(f'{"="*60}')
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        print(f'[FAIL] {tag}')
        failed_runs.append(tag)
    else:
        print(f'[OK] {tag} -> {out_path}')

print(f'\n{"="*60}')
if failed_runs:
    print(f'Failed runs: {failed_runs}')
else:
    print('All ablation runs complete')

## Step 5 — Results table (all strategies across all runs)

In [ ]:
import json, os, glob

result_files = sorted(glob.glob(os.path.join(OUT_DIR, 'ablation_*.json')))

if not result_files:
    print(f'No result files found in {OUT_DIR}')
else:
    for fp in result_files:
        with open(fp) as f: ab = json.load(f)
        run_tag = os.path.basename(fp).replace('ablation_', '').replace('.json', '')
        print(f'\n{"="*70}')
        print(f'RUN: {run_tag}')
        print(f'{"="*70}')
        print(f'{"Strategy":<35} {"F1":>7} {"Sem":>7} {"FFR":>7} {"Mem":>6}')
        print('-' * 60)

        for r in ab.get('results', []):
            strategy = r.get('strategy', '?')
            f1       = r.get('overall_f1', r.get('avg_f1', 0))
            sem      = r.get('avg_semantic_sim', 0)
            ffr      = r.get('false_forgetting_rate', None)
            mem      = r.get('memory_count', 0)
            ffr_str  = f'{ffr:.3f}' if ffr is not None else 'N/A'
            marker   = ' <- OURS' if 'Consolidation-Aware (Ours)' in strategy else ''
            print(f'{strategy:<35} {f1:>7.4f} {sem:>7.4f} {ffr_str:>7} {mem:>6}{marker}')

    print(f'\nTotal runs loaded: {len(result_files)}')
    print('FFR = False Forgetting Rate (0 = no unconsolidated memories evicted)')

## Step 6 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb4_ablation'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')